# Practical Assignment 1: Basic Operations in TensorFlow and PyTorch

**Aim:** Demonstrate the use of TensorFlow, Keras, and PyTorch by implementing fundamental tensor operations, mathematical transformations, automatic differentiation, and simple predictive models in Python.

**Key Concepts to Remember:**
1. **Tensors:** Multi-dimensional arrays with hardware acceleration.
2. **Autograd:** PyTorch uses `requires_grad=True` and `loss.backward()`; TensorFlow uses `with tf.GradientTape() as tape:`.
3. **Keras Sequential Model:** Simple 4-step pipeline: Define Layers $\to$ Compile $\to$ Fit $\to$ Evaluate.


## 1. PyTorch Operations: Creation, Slicing, Math & Autograd


In [1]:
import torch
import numpy as np

# 1.1 Tensor Creation Methods
t_list = torch.tensor([[1.0, 2.0], [3.0, 4.0]])        # from Python list
t_np = torch.from_numpy(np.array([5.0, 6.0]))          # from NumPy array
t_zeros = torch.zeros((2, 2))                          # zeros
t_ones = torch.ones((2, 2))                            # ones
t_rand = torch.rand((2, 2))                            # random uniform [0, 1)

print('PyTorch Tensor from list:\n', t_list)
print('Shape:', t_list.shape, '| Dtype:', t_list.dtype, '| Device:', t_list.device)

# 1.2 Mathematical & Matrix Operations
print('\nAddition (t_list + t_ones):\n', t_list + t_ones)
print('Element-wise Multiply:\n', t_list * 2)
print('Matrix Multiplication (matmul):\n', torch.matmul(t_list, t_list))
print('Transpose:\n', t_list.T)

# 1.3 In-place operation and Slicing
t_list.add_(10.0)  # trailing underscore denotes in-place modification
print('\nIn-place addition (+10):\n', t_list)
print('Slice first row:', t_list[0, :].numpy())

# 1.4 Autograd (Automatic Differentiation): y = 3*w^2 + 2*x + 1
# dy/dw = 6*w -> at w=2.0 => 12.0
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0, requires_grad=True)
y = 3 * (w ** 2) + 2 * x + 1
y.backward()
print(f'\nAutograd dy/dw at w=2.0: {w.grad.item()} (Expected: 12.0)')
print(f'Autograd dy/dx at x=3.0: {x.grad.item()} (Expected: 2.0)')


PyTorch Tensor from list:
 tensor([[1., 2.],
        [3., 4.]])
Shape: torch.Size([2, 2]) | Dtype: torch.float32 | Device: cpu

Addition (t_list + t_ones):
 tensor([[2., 3.],
        [4., 5.]])
Element-wise Multiply:
 tensor([[2., 4.],
        [6., 8.]])
Matrix Multiplication (matmul):
 tensor([[ 7., 10.],
        [15., 22.]])
Transpose:
 tensor([[1., 3.],
        [2., 4.]])

In-place addition (+10):
 tensor([[11., 12.],
        [13., 14.]])
Slice first row: [11. 12.]

Autograd dy/dw at w=2.0: 12.0 (Expected: 12.0)
Autograd dy/dx at x=3.0: 2.0 (Expected: 2.0)


## 2. TensorFlow Operations: Constants, Variables, Math & GradientTape


In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf

# 2.1 Creation: Constants and Variables
c1 = tf.constant([[1.0, 2.0], [3.0, 4.0]])
v1 = tf.Variable([[5.0, 6.0], [7.0, 8.0]])  # mutable weights
tf_zeros = tf.zeros((2, 2))
tf_rand = tf.random.normal((2, 2))

print('TensorFlow Constant:\n', c1.numpy())
print('Shape:', c1.shape, '| Dtype:', c1.dtype)

# 2.2 Mathematical & Matrix Operations
print('\nTF Addition:\n', tf.add(c1, v1).numpy())
print('TF Matrix Multiplication:\n', tf.matmul(c1, v1).numpy())
print('TF Transpose:\n', tf.transpose(c1).numpy())
print('TF Reduce Mean:', tf.reduce_mean(c1).numpy())

# 2.3 Reshaping & Slicing
print('TF Reshaped to (1, 4):', tf.reshape(c1, (1, 4)).numpy())
print('TF Slice Column 0:   ', c1[:, 0].numpy())

# 2.4 GradientTape (Automatic Differentiation): y = 3*w^2 + 2*x + 1
w_tf = tf.Variable(2.0)
x_tf = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y_tf = 3 * (w_tf ** 2) + 2 * x_tf + 1
grads = tape.gradient(y_tf, [w_tf, x_tf])
print(f'\nTF GradientTape dy/dw at w=2.0: {grads[0].numpy()} (Expected: 12.0)')
print(f'TF GradientTape dy/dx at x=3.0: {grads[1].numpy()} (Expected: 2.0)')


TensorFlow Constant:
 [[1. 2.]
 [3. 4.]]
Shape: (2, 2) | Dtype: <dtype: 'float32'>

TF Addition:
 [[ 6.  8.]
 [10. 12.]]
TF Matrix Multiplication:
 [[19. 22.]
 [43. 50.]]
TF Transpose:
 [[1. 3.]
 [2. 4.]]
TF Reduce Mean: 2.5
TF Reshaped to (1, 4): [[1. 2. 3. 4.]]
TF Slice Column 0:    [1. 3.]

TF GradientTape dy/dw at w=2.0: 12.0 (Expected: 12.0)
TF GradientTape dy/dx at x=3.0: 2.0 (Expected: 2.0)


## 3. Keras Predictive Model (from Lab Manual Page 15)


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load Pima Indians Diabetes Dataset
csv_path = r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/pima-indians-diabetes/pima-indians-diabetes.csv'
df = pd.read_csv(csv_path, header=None)
X = df.iloc[:, 0:8].values
y = df.iloc[:, 8].values

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3-Layer Sequential Feedforward Model from Manual
model = tf.keras.Sequential([
    tf.keras.layers.Dense(12, input_shape=(8,), activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile and Fit
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=20, batch_size=16, verbose=1)

# Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\nKeras Diabetes Classification Accuracy: {acc*100:.2f}%')


Epoch 1/20

 1/39 ━━━━━━━━━━━━━━━━━━━━ 24s 642ms/step - accuracy: 0.6250 - loss: 0.6378
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5945 - loss: 0.6960   
Epoch 2/20

 1/39 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6875 - loss: 0.6570
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6661 - loss: 0.6625 
Epoch 3/20

 1/39 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5000 - loss: 0.6585
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6824 - loss: 0.6402 
Epoch 4/20

 1/39 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - loss: 0.7445
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6922 - loss: 0.6183 
Epoch 5/20

 1/39 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8125 - loss: 0.5087
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7003 - loss: 0.5942 
Epoch 6/20

 1/39 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6250 - loss: 0.5824
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7166 - loss: 0.5707 
Epoch 7/20

 1/39 ━━━━━━━━━━━━━━━━━━

C:\Users\Rihan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
